In [15]:
# Please execute/shift-return this cell everytime you run the notebook.  Don't edit it. 
%load_ext autoreload
%autoreload 2
from notebook import * 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Real case studies -- Tegra Orin

## Cache performance of code on "real machines"

### NVIDIA Jetson Nano -- Tegra Orin

In [16]:
render_code("4way_madd/madd.c", show=["//START","//END"])

// 4way_madd/madd.c:55-67 (13 lines)
    //START
    for(i = 0; i < RUN_ARRAY_SIZE; i++)
    {
        e[i] = (a[i] * b[i] + c[i])/d[i];
#ifdef DUMP
        fprintf(stderr, "a[%d], %p\n",i,&a[i]);
        fprintf(stderr, "b[%d], %p\n",i,&b[i]);
        fprintf(stderr, "c[%d], %p\n",i,&c[i]);
        fprintf(stderr, "d[%d], %p\n",i,&d[i]);
        fprintf(stderr, "e[%d], %p\n",i,&e[i]);
#endif
    }
  //END

Let's run it without the above loop code to figure the baseline memory accesses without running the loop on a Jetson nano.

#### Run without the 4-way matrix add loop code.

In [17]:
# Run it "without" the above code.
! ssh htseng@orin-1 "lscpu; cd courses/CS203/demo/memory/4way_madd/; make clean madd_nano; valgrind --tool=cachegrind ./madd_nano 16384 0 "

Architecture:                    aarch64
CPU op-mode(s):                  32-bit, 64-bit
Byte Order:                      Little Endian
CPU(s):                          6
On-line CPU(s) list:             0-5
Thread(s) per core:              1
Core(s) per socket:              6
Socket(s):                       1
Vendor ID:                       ARM
Model:                           1
Model name:                      ARMv8 Processor rev 1 (v8l)
Stepping:                        r0p1
CPU max MHz:                     1971.2000
CPU min MHz:                     115.2000
BogoMIPS:                        62.50
L1d cache:                       384 KiB
L1i cache:                       384 KiB
L2 cache:                        1.5 MiB
L3 cache:                        2 MiB
Vulnerability Itlb multihit:     Not affected
Vulnerability L1tf:              Not affected
Vulnerability Mds:               Not affected
Vulnerability Meltdown:          Not affected
Vulnerability Spec store bypass: Mitigation; S

Too much detail! Let's use grep to narrow down the outputs.

In [18]:
# Run it "without" the above code.
! ssh htseng@orin-1 "cd courses/CS203/demo/memory/4way_madd/; valgrind --tool=cachegrind ./madd_nano 8192 0 >& nano_without_loop.perf; grep 'D   refs\|D1' nano_without_loop.perf"

==12045== D   refs:      1,699,869  (1,069,807 rd   + 630,062 wr)
==12045== D1  misses:        8,697  (    2,854 rd   +   5,843 wr)
==12045== D1  miss rate:       0.5% (      0.3%     +     0.9%  )


Let's run it with the above loop code again and observe the changes in L1 cache misses/accesses

#### Run with the 4-way matrix add loop code.

In [19]:
! ssh htseng@orin-1 "cd courses/CS203/demo/memory/4way_madd/;valgrind --tool=cachegrind ./madd_nano 8192 8192 >& nano_with_loop.perf; grep 'D   refs\|D1' nano_with_loop.perf"

==12097== D   refs:      1,740,835  (1,102,582 rd   + 638,253 wr)
==12097== D1  misses:       49,672  (   35,634 rd   +  14,038 wr)
==12097== D1  miss rate:       2.9% (      3.2%     +     2.2%  )


In [20]:
# Let's do some math here
total_number_of_accesses_before_the_loop =     1699869
total_number_of_accesses_after_the_loop =  1740835
total_number_of_accesses_in_the_loop = total_number_of_accesses_after_the_loop-total_number_of_accesses_before_the_loop
total_number_of_misses_before_the_loop =  8697
total_number_of_misses_after_the_loop = 49672
total_number_of_misses_in_the_loop = total_number_of_misses_after_the_loop-total_number_of_misses_before_the_loop
miss_rate_of_the_loop = total_number_of_misses_in_the_loop/total_number_of_accesses_in_the_loop

print(f"access in the loop: %d misses in the loop %d miss_rate %lf" % (total_number_of_accesses_in_the_loop, total_number_of_misses_in_the_loop, miss_rate_of_the_loop))

access in the loop: 40966 misses in the loop 40975 miss_rate 1.000220


In [8]:
! cd 4way_madd; make clean madd_dump; cd ..; 
!echo "element,address"> addresses_madd.csv; 
!./4way_madd/madd_dump 8192 8192 2>> addresses_madd.csv
! head -n 101 addresses_madd.csv > addresses_digest.csv
df = pd.read_csv("addresses_digest.csv",skipfooter=1,engine='python')
df["address"] = df["address"].str.replace('0x','')
df["address"]=df[["address"]].apply(lambda x: x.astype(str).map(lambda x: int(x, base=16)))
# only show the first N addresses 
#N = 32
#df2 = df2.iloc[:N]
C = 32768
B = 64
A = 4
offset_bits = int(math.log2(B))
S = int(C/(B*A))
index_bits = int(math.log2(S))
df["tag"]=(df["address"].apply(lambda x: x >> (offset_bits+index_bits)))
df["tag"] = df["tag"].apply(lambda x: hex(x))
df["index"] = df["address"].apply(lambda x: hex((x>>offset_bits)%S))
df["address"] = df["address"].apply(lambda x: hex(x))
display_df_mono(df)

rm -f madd_intel madd_nano *_O3 *~ madd_A_fission cachegrind* *.perf madd_dump
cc -O3 -DHAVE_LINUX_PERF_EVENT_H -g  -DDUMP perfstats.c madd.c -o madd_dump
78030155,32259828,0.413428,0.590301,0.019043,0.000162,5445,33510511


,element,address,tag,index
0,a[0],0x701ac90cf000,0x380d64867,0x40
1,b[0],0x701ac90df000,0x380d6486f,0x40
2,c[0],0x701ac90ef000,0x380d64877,0x40
3,d[0],0x701ac90ff000,0x380d6487f,0x40
4,e[0],0x701ac910f000,0x380d64887,0x40
5,a[1],0x701ac90cf008,0x380d64867,0x40
6,b[1],0x701ac90df008,0x380d6486f,0x40
7,c[1],0x701ac90ef008,0x380d64877,0x40
8,d[1],0x701ac90ff008,0x380d6487f,0x40
9,e[1],0x701ac910f008,0x380d64887,0x40


In [9]:
display_df_mono(render_csv("./4way_madd/4way_add.csv"))

,index,CPU,IC,Cycles,CPI,CT,ET,miss_rate,misses,accesses
0,0,Jetson Orin,191466,93652,0.489131,1.185239,0.000111,0.002591,222,85690


In [13]:
! echo "CPU, IC, Cycles, CPI, CT, ET, miss_rate, misses, accesses" >& ./4way_madd/4way_add.csv
! echo -n "Jetson Orin," >> ./4way_madd/4way_add.csv
! ssh htseng@orin-1 "cd courses/CS203/demo/memory/4way_madd/; make madd_nano ;./madd_nano 131072 131072 >> 4way_add.csv"

make: 'madd_nano' is up to date.


In [14]:
! cat ./4way_madd/4way_add.csv

CPU, IC, Cycles, CPI, CT, ET, miss_rate, misses, accesses
Jetson Orin,1453034,667595,0.459449,1.337637,0.000893,0.000369,243,659012
